In [ ]:
# Add at the very top of your file, before other imports
import os
import subprocess
import tempfile
import re
import itertools
from typing import List, Optional
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import httpx
import requests
from pydub.effects import normalize

# Install required system packages and configure ImageMagick
!apt-get update
!apt-get install -y imagemagick fonts-dejavu fonts-liberation firefox
!pip uninstall -y moviepy  # Uninstall existing version
!pip install moviepy==1.0.3 python-telegram-bot pydub yt-dlp librosa scipy numpy requests fonttools nest_asyncio beautifulsoup4 httpx selenium
!apt-get update
!apt install -y chromium-chromedriver

# Configure ImageMagick policy to allow text operations
def configure_imagemagick():
    """Configure ImageMagick with proper security policy."""
    try:
        # Create a more permissive policy file
        policy_xml = '''<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE policymap [
<!ELEMENT policymap (policy)+>
<!ELEMENT policy EMPTY>
<!ATTLIST policy domain (delegate|coder|filter|path|resource) #IMPLIED>
<!ATTLIST policy name CDATA #IMPLIED>
<!ATTLIST policy rights CDATA #IMPLIED>
<!ATTLIST policy pattern CDATA #IMPLIED>
<!ATTLIST policy value CDATA #IMPLIED>
]>
<policymap>
  <policy domain="resource" name="memory" value="2GiB"/>
  <policy domain="resource" name="map" value="4GiB"/>
  <policy domain="resource" name="width" value="16KP"/>
  <policy domain="resource" name="height" value="16KP"/>
  <policy domain="resource" name="area" value="128MB"/>
  <policy domain="resource" name="disk" value="8GiB"/>
  <policy domain="delegate" rights="none" pattern="URL" />
  <policy domain="delegate" rights="none" pattern="HTTPS" />
  <policy domain="delegate" rights="none" pattern="HTTP" />
  <policy domain="path" rights="none" pattern="@*"/>
  <policy domain="cache" name="shared-secret" value="passphrase" stealth="true"/>
  <!-- Allow text operations -->
  <policy domain="path" rights="read|write" pattern="@*"/>
  <policy domain="coder" rights="read|write" pattern="*"/>
</policymap>'''

        # Write the policy file
        with open('/etc/ImageMagick-6/policy.xml', 'w') as f:
            f.write(policy_xml)

        # Set proper permissions
        os.system('chmod 644 /etc/ImageMagick-6/policy.xml')

        # Configure MoviePy to use ImageMagick
        os.environ['IMAGEMAGICK_BINARY'] = '/usr/bin/convert'

        # Test configuration with tiny size
        from moviepy.editor import TextClip
        test = TextClip("Test", fontsize=32, color='white', method='label').set_duration(1)
        test.close()

        print("ImageMagick configured successfully")
        return True
    except Exception as e:
        print(f"Error configuring ImageMagick: {e}")
        return False

# Initialize font system
def setup_fonts_and_imagemagick():
    """Setup fonts and ImageMagick configuration."""
    try:
        # Install additional fonts
        subprocess.run(['apt-get', 'install', '-y', 'fonts-dejavu', 'fonts-liberation'],
                      stdout=subprocess.DEVNULL,
                      stderr=subprocess.DEVNULL)

        # Create Firefox profile for cookies
        subprocess.run(['mkdir', '-p', '/root/.mozilla/firefox'])
        subprocess.run(['firefox', '-CreateProfile', 'default'])

        # Update font cache
        subprocess.run(['fc-cache', '-f', '-v'],
                      stdout=subprocess.DEVNULL,
                      stderr=subprocess.DEVNULL)

        # Configure ImageMagick
        if not configure_imagemagick():
            return False

        # Test text creation with tiny size
        from moviepy.editor import TextClip
        test_clip = TextClip("Test", fontsize=32, method='label')
        test_clip.close()

        return True
    except Exception as e:
        print(f"Error setting up environment: {e}")
        return False

# Run setup
if not setup_fonts_and_imagemagick():
    print("Failed to configure environment. Text rendering may not work properly.")

# final edit 2tempfile
import logging
import os
import tempfile
from typing import List, Optional
from functools import wraps
import json
import hashlib
import random
import asyncio
import io
from telegram.error import TimedOut, NetworkError, RetryAfter
import warnings

# import edge_tts
import sys

# Add nest_asyncio import at the top with other imports
import nest_asyncio

try:
    from moviepy.editor import (
        VideoFileClip,
        TextClip,
        CompositeVideoClip,
        concatenate_videoclips,
        AudioFileClip,
        CompositeAudioClip
    )
    from moviepy.video.fx.all import resize, crop, time_mirror, fadein, fadeout
    from moviepy.config import get_setting
    from moviepy.audio.io.AudioFileClip import AudioFileClip
    from moviepy.audio.AudioClip import CompositeAudioClip, concatenate_audioclips
except ImportError as e:
    print(f"Error importing moviepy components: {e}")
    print("Please run: !pip install moviepy==1.0.3")
    sys.exit(1)

from pydub import AudioSegment
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup, InputMediaPhoto
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    CallbackContext,
    filters,
    CallbackQueryHandler
)
import numpy as np
from scipy.signal import find_peaks
import librosa
from yt_dlp import YoutubeDL
import re
from telegram.error import TimedOut, NetworkError, RetryAfter
import warnings

# Add Google Drive imports and setup
from google.colab import drive
import time
from datetime import datetime, timedelta

# Mount Google Drive and setup storage
try:
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/telegram_bot/"
    video_storage = os.path.join(BASE_PATH, "videos")
    CREDITS_FILE = os.path.join(BASE_PATH, "user_credits.json")
except Exception as e:
    print(f"Failed to mount Google Drive: {e}")
    BASE_PATH = "/content"
    video_storage = os.path.join(BASE_PATH, "videos")
    CREDITS_FILE = os.path.join(BASE_PATH, "user_credits.json")

# Create necessary directories
for directory in [video_storage]:
    if not os.path.exists(directory):
        os.makedirs(directory)

# Setup TTS storage
TTS_STORAGE = os.path.join(BASE_PATH, "tts")
os.makedirs(TTS_STORAGE, exist_ok=True)

# Setup logging
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

# Add these constants after the logger setup
ADMIN_PASSWORD_HASH = hashlib.sha256("your_admin_password_here".encode()).hexdigest()  # Change this to your desired admin password
MAX_VIDEO_DURATION = 60  # Maximum duration in seconds for any video clip
FRAME_BUFFER_SIZE = 128  # Number of frames to buffer

# Add error handling constants
ERROR_MESSAGES = {
    "server_error": "🔧 Our servers are currently under maintenance. Please try again in a few minutes.",
    "video_error": "📹 There was an issue with your video. Please make sure it's in MP4 format and try again.",
    "audio_error": "🎵 There was an issue with your audio. Please make sure it's in MP3 format and try again.",
    "processing_error": "⚠️ Video processing failed. This might be due to:\n• Video format not supported\n• File too large\n• Server busy\nPlease try again or contact support.",
    "quota_exceeded": "📊 Our servers are currently at capacity. Please try again in a few minutes.",
    "file_too_large": "📁 File size too large. Please keep videos under 50MB.",
    "memory_error": "🚫 Video processing stopped due to memory limits.\nTry:\n• Shorter video clips\n• Lower resolution videos\n• Fewer video clips",
    "killed_error": "⚠️ Process terminated unexpectedly.\nTry:\n• Reducing video length\n• Using smaller files\n• Processing fewer clips at once",
}

# Add these constants after other constants
SCENE_DETECTION_THRESHOLD = 10  # More sensitive detection
MIN_SCENE_DURATION = 0.4       # Slightly longer minimum duration
MAX_RETRIES = 3
RETRY_DELAY = 5  # seconds

# Add these constants after other constants
DEFAULT_FONT = 'Times-Roman'
ALLOWED_FONT_EXTENSIONS = ('.ttf', '.otf')

# Add these constants for TTS
TTS_BASE_URL = "https://lazypy.ro/tts/"
TTS_GENERATE_URL = urljoin(TTS_BASE_URL, "speak")  # The actual endpoint might be different
VOICE_CACHE = {}  # Cache for available voices

# Add new function to handle font file uploads
async def handle_font_file(update: Update, context: CallbackContext) -> None:
    """Handle uploaded font files."""
    try:
        if not update.message.document:
            await update.message.reply_text("Please send a TTF or OTF font file.")
            return

        file = await update.message.document.get_file()
        file_ext = os.path.splitext(file.file_path)[1].lower()

        if file_ext not in ALLOWED_FONT_EXTENSIONS:
            await update.message.reply_text(
                "❌ Invalid font format!\n"
                "Please send a TTF or OTF font file."
            )
            return

        # Create fonts directory if it doesn't exist
        font_dir = os.path.join(video_storage, "fonts")
        os.makedirs(font_dir, exist_ok=True)

        # Save font file with user ID prefix
        font_path = os.path.join(
            font_dir,
            f"font_{update.message.from_user.id}{file_ext}"
        )

        await file.download_to_drive(font_path)

        # Verify font file is valid
        try:
            test_clip = TextClip("Test", font=font_path, fontsize=32, method='label')
            test_clip.close()

            # Store font path in user data
            context.user_data['text_font'] = font_path

            await update.message.reply_text(
                "✅ Font uploaded successfully!\n"
                "This font will be used for your next video."
            )
        except Exception as e:
            os.remove(font_path)
            logger.error(f"Invalid font file: {e}")
            await update.message.reply_text(
                "❌ Invalid font file!\n"
                "Please send a valid TTF or OTF font file."
            )

    except Exception as e:
        logger.error(f"Error handling font file: {e}")
        await update.message.reply_text(
            "❌ Error processing font file.\n"
            "Please try again or use the default font."
        )

# Update the show_fonts function to provide a choice
async def show_fonts(update: Update, context: CallbackContext) -> None:
    """Show font options to user."""
    keyboard = [
        [InlineKeyboardButton("Use Default Font (Times-Roman)", callback_data="font_default")],
        [InlineKeyboardButton("Upload Custom Font (TTF/OTF)", callback_data="font_upload")]
    ]
    reply_markup = InlineKeyboardMarkup(keyboard)

    await update.message.reply_text(
        "🎨 Choose your font option:\n\n"
        "1. Use the default Times-Roman font\n"
        "2. Upload your own TTF/OTF font file",
        reply_markup=reply_markup
    )

# Add font callback handler
async def font_callback(update: Update, context: CallbackContext) -> None:
    """Handle font choice callback."""
    query = update.callback_query
    await query.answer()

    if query.data == "font_default":
        context.user_data['text_font'] = 'Times-Roman'
        await query.edit_message_text(
            "✅ Default font (Times-Roman) will be used.\n"
            "You can proceed with creating your video!"
        )
    elif query.data == "font_upload":
        await query.edit_message_text(
            "📤 Please send your TTF or OTF font file.\n\n"
            "Note: The file should be a valid font file."
        )

def is_youtube_shorts_url(url: str) -> bool:
    """Check if the URL is a valid YouTube Shorts link."""
    youtube_shorts_pattern = r'(https?://)?(www\.)?(youtube\.com/shorts/|youtu\.be/)[A-Za-z0-9_-]+'
    return bool(re.match(youtube_shorts_pattern, url))

# Move these utility functions to the top, before any handler functions
def verify_font_file(font_path: str) -> bool:
    """Verify that a font file is valid and usable."""
    try:
        # Try to load the font using fontTools
        font = TTFont(font_path)
        font.close()

        # Test with MoviePy
        test_clip = TextClip("Test", font=font_path, fontsize=32, method='label')
        test_clip.close()

        return True
    except Exception as e:
        logger.error(f"Font verification failed: {e}")
        return False

def verify_font_availability(font_name: str) -> str:
    """Verify font availability and return valid font name."""
    try:
        # Test the font
        test_clip = TextClip("Test", font=font_name, fontsize=32, method='label')
        test_clip.close()
        return font_name
    except:
        logger.warning(f"Font '{font_name}' not available, falling back to Times-Roman")
        return 'Times-Roman'

def process_quote(text: str) -> List[str]:
    """Process the quote into chunks for display."""
    try:
        # Clean up the text
        text = text.strip()

        # For very short phrases (3 words or less), return as single chunk
        words = text.split()
        if len(words) <= 3:
            return [text]

        # Rest of the processing for longer quotes
        chunks = []
        current_chunk = []
        current_length = 0
        max_length = 50  # Maximum characters per chunk

        for word in words:
            # Check if adding this word would exceed max length
            if current_length + len(word) + 1 > max_length and current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                current_length = 0

            current_chunk.append(word)
            current_length += len(word) + 1

            # If we have 3 words, create a chunk (for three_words_mode)
            if len(current_chunk) >= 3:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                current_length = 0

        # Add any remaining words
        if current_chunk:
            chunks.append(' '.join(current_chunk))

        return chunks if chunks else [text]  # Fallback to original text if no chunks created

    except Exception as e:
        logger.error(f"Error processing quote: {e}")
        return [text]  # Return original text as single chunk on error

def create_text_clip(text: str, duration: float, font: str = 'Times-Roman',
                    font_size: int = 48) -> TextClip:
    """Create centered text clip with proper synchronization."""
    try:
        # Create text clip with center alignment
        txt_clip = TextClip(
            text,
            fontsize=font_size,
            color='white',
            font=font,
            align='center',
            method='caption',  # Better for multi-line text
            size=(720, None)  # Width matches video width (720px for vertical video)
        ).set_duration(duration)

        # Center vertically and horizontally
        txt_clip = txt_clip.set_position(('center', 'center'))

        return txt_clip
    except Exception as e:
        logger.error(f"Error creating text clip: {e}")
        return TextClip(" ", size=(0,0)).set_duration(0.1)  # Return empty clip as fallback

def load_credits() -> dict:
    """Load credits from file."""
    try:
        if os.path.exists(CREDITS_FILE):
            with open(CREDITS_FILE, 'r') as f:
                return json.load(f)
    except Exception as e:
        logger.error(f"Error loading credits: {e}")
    return {}

def save_credits(credits: dict) -> None:
    """Save credits to file."""
    try:
        with open(CREDITS_FILE, 'w') as f:
            json.dump(credits, f)
    except Exception as e:
        logger.error(f"Error saving credits: {e}")

def check_credits(func):
    """Decorator to check if user has enough credits."""
    @wraps(func)
    async def wrapper(update: Update, context: CallbackContext, *args, **kwargs):
        user_id = str(update.message.from_user.id)
        credits = load_credits()
        current_credits = credits.get(user_id, 0)

        if current_credits <= 0:
            await update.message.reply_text(
                "❌ You don't have enough credits!\n"
                "Please contact the administrator to get more credits."
            )
            return

        return await func(update, context, *args, **kwargs)
    return wrapper

@check_credits
async def handle_video(update: Update, context: CallbackContext) -> None:
    """Handle incoming video files."""
    video_path = None
    try:
        # Check file size
        if update.message.video.file_size > 20 * 1024 * 1024:  # 20MB limit (Telegram Bot API limit)
            await update.message.reply_text(ERROR_MESSAGES["file_too_large"])
            return

        video_file = await update.message.video.get_file()
        video_path = os.path.join(
            video_storage,
            f"{update.message.from_user.id}_video_{len(context.user_data.get('video_paths', []))}.mp4"
        )

        await video_file.download_to_drive(video_path)

        # Validate video before adding to user_data
        if not is_valid_video(video_path):
            os.remove(video_path)
            await update.message.reply_text(
                "⚠️ This video seems to be corrupted or in an unsupported format.\n"
                "Please try another video file."
            )
            return

        if 'video_paths' not in context.user_data:
            context.user_data['video_paths'] = []
        context.user_data['video_paths'].append(video_path)

        await update.message.reply_text(
            "✅ Video received!\n"
            "• Send more videos or\n"
            "• Share your quote 💭"
        )
    except Exception as e:
        logger.error(f"Video handling error: {str(e)}")
        if video_path and os.path.exists(video_path):
            os.remove(video_path)
        if "connection" in str(e).lower():
            await update.message.reply_text(ERROR_MESSAGES["server_error"])
        else:
            await update.message.reply_text(ERROR_MESSAGES["video_error"])

async def fetch_available_voices():
    """Fetch available voices from lazypy.ro/tts/"""
    try:
        # Only StreamElements voices
        voices = {
            "Brian": "Brian (English, British) 🇬🇧",
            "Amy": "Amy (English, British) 🇬🇧",
            "Emma": "Emma (English, British) 🇬🇧",
            "Joey": "Joey (English, American) 🇺🇸",
            "Justin": "Justin (English, American) 🇺🇸",
            "Russell": "Russell (English, Australian) 🇦🇺",
            "Nicole": "Nicole (English, Australian) 🇦🇺",
            "Geraint": "Geraint (English, Welsh) 🏴󠁧󠁢󠁷󠁬󠁳󠁿"
        }

        logger.info(f"Found {len(voices)} StreamElements voices")
        return voices
    except Exception as e:
        logger.error(f"Error fetching voices: {e}")
        return {}

async def generate_tts_audio(text: str, voice_id: str) -> Optional[str]:
    """Generate TTS audio using Selenium to interact with the TTS service."""
    try:
        # Configure Selenium options
        from selenium import webdriver
        from selenium.webdriver.chrome.service import Service
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        from selenium.common.exceptions import TimeoutException, WebDriverException

        # Setup Chrome options
        chrome_options = Options()
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--headless=new')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--window-size=1920,1080')
        chrome_options.add_argument('--disable-web-security')
        chrome_options.add_argument('--allow-running-insecure-content')
        chrome_options.add_argument('--disable-features=IsolateOrigins,site-per-process')

        # Create service
        service = Service()

        logger.info("Starting Chrome...")
        driver = webdriver.Chrome(service=service, options=chrome_options)

        try:
            # Set longer timeouts
            driver.set_page_load_timeout(45)
            wait = WebDriverWait(driver, 45)

            # 1. Go to the TTS website
            logger.info("Loading TTS website...")
            driver.get("https://lazypy.ro/tts/")

            # Wait for page to load completely
            wait.until(lambda d: d.execute_script('return document.readyState') == 'complete')

            # 2. Find and fill the text input
            logger.info("Entering text...")
            text_area = wait.until(EC.presence_of_element_located((By.TAG_NAME, "textarea")))
            driver.execute_script("arguments[0].value = arguments[1]", text_area, text)

            # 3. Select StreamElements service
            logger.info("Selecting service...")
            driver.execute_script("""
                var selects = document.getElementsByTagName('select');
                for (var i = 0; i < selects.length; i++) {
                    if (selects[i].options[0].text.includes('StreamElements')) {
                        selects[i].value = 'StreamElements';
                        var event = new Event('change', { bubbles: true });
                        selects[i].dispatchEvent(event);
                        break;
                    }
                }
            """)

            # Wait for voice options to load
            await asyncio.sleep(3)

            # 4. Select voice with retry
            logger.info(f"Selecting voice: {voice_id}")
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    driver.execute_script(f"""
                        var selects = document.getElementsByTagName('select');
                        for (var i = 0; i < selects.length; i++) {{
                            for (var j = 0; j < selects[i].options.length; j++) {{
                                if (selects[i].options[j].text.includes('{voice_id}')) {{
                                    selects[i].selectedIndex = j;
                                    var event = new Event('change', {{ bubbles: true }});
                                    selects[i].dispatchEvent(event);
                                    break;
                                }}
                            }}
                        }}
                    """)
                    break
                except Exception as e:
                    if attempt == max_retries - 1:
                        raise
                    await asyncio.sleep(1)

            # 5. Click "Say It" button
            logger.info("Clicking Say It button...")
            say_it = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Say It')]")))
            say_it.click()

            # 6. Wait for audio to be generated
            logger.info("Waiting for audio...")
            audio = None
            for attempt in range(max_retries):
                try:
                    audio = wait.until(EC.presence_of_element_located((By.TAG_NAME, "audio")))
                    break
                except TimeoutException:
                    if attempt == max_retries - 1:
                        raise
                    driver.refresh()
                    await asyncio.sleep(2)

            if not audio:
                raise Exception("Audio element not found")

            # 7. Get the audio URL
            audio_url = audio.get_attribute("src")
            if not audio_url:
                raise Exception("Audio URL not found")

            # 8. Download the audio file
            logger.info("Downloading audio...")
            response = requests.get(audio_url, timeout=30)
            if response.status_code != 200:
                raise Exception(f"Failed to download audio: HTTP {response.status_code}")

            # Save the audio file
            file_hash = hashlib.md5(text.encode()).hexdigest()
            audio_filename = f"tts_{file_hash}.mp3"
            audio_path = os.path.join(TTS_STORAGE, audio_filename)

            os.makedirs(os.path.dirname(audio_path), exist_ok=True)
            with open(audio_path, 'wb') as f:
                f.write(response.content)

            # Verify the audio file
            if os.path.getsize(audio_path) < 1024:  # Less than 1KB
                raise Exception("Generated audio file is too small")

            logger.info(f"TTS audio saved to: {audio_path}")
            return audio_path

        finally:
            try:
                driver.quit()
            except:
                pass

    except Exception as e:
        logger.error(f"TTS generation failed: {str(e)}")
        import traceback
        logger.error(f"Full traceback: {traceback.format_exc()}")
        return None

# Modify handle_text function to include voice selection
async def handle_text(update: Update, context: CallbackContext) -> None:
    """Handle incoming text messages."""
    if is_youtube_shorts_url(update.message.text):
        await handle_music(update, context)
        return

    # Handle initial quote
    if not any([
        context.user_data.get('awaiting_reverse_choice'),
        context.user_data.get('awaiting_text_choice'),
        context.user_data.get('awaiting_voice_choice')
    ]):
        try:
            user_text = update.message.text
            context.user_data["quote_text"] = user_text  # Store full quote
            context.user_data["quote_chunks"] = process_quote(user_text)

            # Fetch available voices
            voices = await fetch_available_voices()
            if voices:
                voice_list = "\n".join([f"• {name}: {desc}" for name, desc in voices.items()])
                await update.message.reply_text(
                    "✅ Quote received!\n"
                    f"'{user_text}'\n\n"
                    "🎙️ Choose a voice for your quote:\n\n"
                    f"{voice_list}\n\n"
                    "Type the voice name (e.g., 'Brian') or 'skip' to continue without voiceover"
                )
                context.user_data['available_voices'] = voices
                context.user_data['awaiting_voice_choice'] = True
            else:
                await update.message.reply_text(
                    "❌ Failed to fetch voices. Continuing without voiceover.\n"
                    "Would you like to add reverse effects to your video?\n"
                    "Reply with 'yes' or 'no'"
                )
                context.user_data['awaiting_reverse_choice'] = True
            return

        except Exception as e:
            logger.error(f"Error processing quote: {e}")
            await update.message.reply_text("❌ Something went wrong. Please try again.")
            return

    # Handle voice selection
    elif context.user_data.get('awaiting_voice_choice'):
        voice_choice = update.message.text.strip()
        if voice_choice.lower() == 'skip':
            context.user_data['awaiting_voice_choice'] = False
            await update.message.reply_text(
                "Would you like to add reverse effects to your video?\n"
                "Reply with 'yes' or 'no'"
            )
            context.user_data['awaiting_reverse_choice'] = True
            return

        # Case-insensitive voice matching
        available_voices = context.user_data.get('available_voices', {})
        matched_voice = next(
            (name for name in available_voices.keys()
             if name.lower() == voice_choice.lower()),
            None
        )

        if matched_voice:
            await update.message.reply_text("⏳ Generating voiceover...")

            # Generate voiceover for each chunk
            text_chunks = context.user_data.get('quote_chunks', [])
            voice_paths = []

            for chunk in text_chunks:
                audio_path = await generate_tts_audio(chunk, matched_voice)
                if audio_path:
                    voice_paths.append(audio_path)
                else:
                    await update.message.reply_text("❌ Failed to generate some voiceovers. Continuing without voice.")
                    voice_paths = []
                    break

            context.user_data['voice_paths'] = voice_paths
            await update.message.reply_text(
                "Would you like to add reverse effects to your video?\n"
                "Reply with 'yes' or 'no'"
            )
            context.user_data['awaiting_voice_choice'] = False
            context.user_data['awaiting_reverse_choice'] = True
        else:
            await update.message.reply_text("❌ Invalid voice selection. Please choose from the list or type 'skip'")
        return

    # Handle reverse effect choice
    if context.user_data.get('awaiting_reverse_choice'):
        choice = update.message.text.lower()
        if choice not in ['yes', 'no']:
            await update.message.reply_text("Please reply with 'yes' or 'no'")
            return

        # Store the choice explicitly as a boolean
        context.user_data['use_reverse_effect'] = (choice == 'yes')
        logger.info(f"Reverse effect set to: {context.user_data['use_reverse_effect']}")
        context.user_data['awaiting_reverse_choice'] = False

        # Ask about text display preference
        await update.message.reply_text(
            "How would you like the text to appear?\n"
            "1️⃣ Static (entire quote stays fixed)\n"
            "2️⃣ Sequential mode\n"
            "3️⃣ Three words mode\n"
            "Reply with '1', '2', or '3'"
        )
        context.user_data['awaiting_text_choice'] = True
        return

    # Handle text display choice
    elif context.user_data.get('awaiting_text_choice'):
        choice = update.message.text.strip()
        if choice not in ['1', '2', '3']:
            await update.message.reply_text("Please reply with '1', '2', or '3'")
            return

        context.user_data['static_text'] = (choice == '1')
        context.user_data['process_quote'] = (choice == '2')
        context.user_data['three_words_mode'] = (choice == '3')
        context.user_data['awaiting_text_choice'] = False

        # Go straight to asking for music
        await update.message.reply_text(
            "✅ Preferences saved!\n"
            "🎵 Now send your background music [MP3 or YouTube Shorts link]"
        )
        return

def is_valid_video(video_path: str) -> bool:
    """Check if video file is valid and can be processed."""
    try:
        clip = VideoFileClip(video_path, audio=False)
        if clip.duration < 0.1:  # Check for corrupted video
            clip.close()
            return False
        clip.close()
        return True
    except Exception as e:
        logger.error(f"Video validation error for {video_path}: {e}")
        return False

@check_credits
async def handle_music(update: Update, context: CallbackContext) -> None:
    """Handle background music input."""
    try:
        if update.message.text and is_youtube_shorts_url(update.message.text):
            await update.message.reply_text("⏳ Downloading audio from YouTube Shorts...")
            music_path = await download_youtube_shorts(update.message.text)
            if not music_path:
                await update.message.reply_text(
                    "❌ Failed to download audio from YouTube Shorts.\n"
                    "Please try another link or send an MP3 file directly."
                )
                return
        elif update.message.audio:
            if update.message.audio.file_size > 20 * 1024 * 1024:  # 20MB limit
                await update.message.reply_text(ERROR_MESSAGES["file_too_large"])
                return

            music_file = await update.message.audio.get_file()
            music_path = os.path.join(video_storage, f"{update.message.from_user.id}_music.mp3")
            await music_file.download_to_drive(music_path)
        else:
            await update.message.reply_text(
                "Please send either:\n"
                "• An MP3 file 🎵\n"
                "• A YouTube Shorts link 🔗"
            )
            return

        # Validate music file
        try:
            music = AudioSegment.from_file(music_path)
            if len(music) < 5000:  # Less than 5 seconds
                await update.message.reply_text("❌ Audio file too short. Please use a longer track.")
                return
            if len(music) > 300000:  # More than 5 minutes
                await update.message.reply_text("❌ Audio file too long. Please use a shorter track (max 5 minutes).")
                return
        except Exception as e:
            logger.error(f"Error validating music file: {e}")
            await update.message.reply_text(ERROR_MESSAGES["audio_error"])
            if os.path.exists(music_path):
                os.remove(music_path)
            return

        # Generate beat-synchronized video
        await generate_final_video(update, context, music_path)

    except Exception as e:
        logger.error(f"Music handling error: {str(e)}")
        await update.message.reply_text(ERROR_MESSAGES["audio_error"])
        if 'music_path' in locals() and os.path.exists(music_path):
            os.remove(music_path)

def validate_audio_file(audio_path: str) -> bool:
    """Validate audio file using ffmpeg."""
    try:
        # First try to read with ffprobe for detailed info
        probe_result = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', audio_path],
            capture_output=True,
            text=True
        )

        if probe_result.returncode != 0:
            logger.error(f"FFprobe validation failed for {audio_path}")
            return False

        # Try to read the audio file using pydub with explicit format
        try:
            audio = AudioSegment.from_file(audio_path, format="mp3", parameters=["-acodec", "mp3"])
        except:
            try:
                # Fallback to automatic format detection
                audio = AudioSegment.from_file(audio_path)
            except Exception as e:
                logger.error(f"Pydub failed to read audio {audio_path}: {e}")
                return False

        # Check if audio has valid duration and channels
        if len(audio) <= 0:
            logger.error(f"Invalid audio duration in {audio_path}")
            return False

        return True
    except Exception as e:
        logger.error(f"Audio validation error for {audio_path}: {e}")
        return False

def ensure_valid_audio(audio_path: str) -> str:
    """Ensure audio file is valid by re-encoding if necessary."""
    try:
        if validate_audio_file(audio_path):
            return audio_path

        # If validation fails, try to re-encode the audio
        logger.info(f"Re-encoding audio file: {audio_path}")
        temp_path = audio_path + '.temp.mp3'

        try:
            # First attempt: Basic re-encoding
            subprocess.run([
                'ffmpeg', '-y', '-i', audio_path,
                '-acodec', 'libmp3lame',
                '-ar', '44100',
                '-ac', '2',
                '-b:a', '192k',
                temp_path
            ], check=True, capture_output=True)

            if validate_audio_file(temp_path):
                os.replace(temp_path, audio_path)
                return audio_path

            # Second attempt: More aggressive re-encoding
            subprocess.run([
                'ffmpeg', '-y', '-i', audio_path,
                '-acodec', 'libmp3lame',
                '-ar', '44100',
                '-ac', '2',
                '-b:a', '192k',
                '-write_xing', '0',  # Disable Xing header
                '-id3v2_version', '0',  # Disable ID3 tags
                temp_path
            ], check=True, capture_output=True)

            if validate_audio_file(temp_path):
                os.replace(temp_path, audio_path)
                return audio_path

        except subprocess.CalledProcessError as e:
            logger.error(f"FFmpeg re-encoding failed: {e.stderr}")
            raise

        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise Exception("Failed to create valid audio file after multiple re-encoding attempts")

    except Exception as e:
        logger.error(f"Error ensuring valid audio: {e}")
        if 'temp_path' in locals() and os.path.exists(temp_path):
            os.remove(temp_path)
        raise

# Update the audio processing part in generate_final_video
async def generate_final_video(update: Update, context: CallbackContext, music_path: str) -> None:
    """Generate final video with randomized clips at every beat, synchronized text, and voiceover."""
    loading_msg = await update.message.reply_text("🎶 Syncing videos with music beats...")

    try:
        # Load music
        music_clip = AudioFileClip(music_path)
        music_duration = music_clip.duration

        # Load voiceover if available
        voice_clips = []
        if "voice_paths" in context.user_data:
            for voice_path in context.user_data["voice_paths"]:
                try:
                    voice_clip = AudioFileClip(voice_path)
                    voice_clips.append(voice_clip)
                except Exception as e:
                    logger.error(f"Error loading voiceover {voice_path}: {e}")

        # Load video clips
        video_clips = []
        for video_path in context.user_data.get('video_paths', []):
            try:
                clip = VideoFileClip(video_path, audio=False)
                clip = resize(clip, height=1280)
                clip = crop(clip, width=720, height=1280)
                video_clips.append(clip)
            except Exception as e:
                logger.error(f"Error processing {video_path}: {e}")

        if not video_clips:
            await update.message.reply_text("❌ No valid video clips found!")
            return

        # Detect beats in the music
        beat_times = improved_beat_detect(music_path)
        if not beat_times:
            logger.warning("No beats detected. Using fallback intervals.")
            beat_times = np.arange(0, music_duration, 1.0).tolist()  # Fallback to 1-second intervals
        else:
            # Ensure beat_times is a list
            if isinstance(beat_times, np.ndarray):
                beat_times = beat_times.tolist()

            # Ensure start is 0
            if beat_times[0] > 0.1:
                beat_times.insert(0, 0.0)
            else:
                beat_times[0] = 0.0

            # Ensure end is music_duration
            if beat_times[-1] < music_duration - 0.1:
                beat_times.append(music_duration)
            else:
                beat_times[-1] = music_duration

        # Randomize video clips for each beat
        final_video_segments = []
        available_clips = video_clips.copy()
        last_clip = None

        for i in range(len(beat_times) - 1):
            try:
                start_time = beat_times[i]
                end_time = beat_times[i + 1]
                segment_duration = end_time - start_time

                # Shuffle available clips if empty
                if not available_clips:
                    available_clips = video_clips.copy()
                    random.shuffle(available_clips)

                # Get next clip (ensure it's different from last if possible)
                current_clip = available_clips.pop()
                if current_clip is last_clip and available_clips:
                    available_clips.append(current_clip)
                    current_clip = available_clips.pop()
                last_clip = current_clip

                # Create segment
                max_start = max(0, current_clip.duration - segment_duration - 0.1)
                segment_start = random.uniform(0, max_start)
                segment_end = min(current_clip.duration, segment_start + segment_duration)

                segment = current_clip.subclip(segment_start, segment_end)

                # Ensure segment matches exactly the required duration
                if segment.duration < segment_duration:
                    # If clip is too short, loop it to fill the gap
                    loops_needed = int(segment_duration / segment.duration) + 1
                    segment = concatenate_videoclips([segment] * loops_needed)
                    segment = segment.subclip(0, segment_duration)

                final_video_segments.append(segment)

            except Exception as e:
                logger.error(f"Error creating segment at beat {i}: {e}")
                continue

        if not final_video_segments:
            raise Exception("No valid video segments created")

        # Concatenate video segments
        final_video = concatenate_videoclips(final_video_segments, method="compose")
        final_video = final_video.set_duration(music_duration)

        # Create text overlays with simple white text
        text_clips = []
        current_start = 0
        if "quote_chunks" in context.user_data:
            for i, chunk in enumerate(context.user_data["quote_chunks"]):
                try:
                    # Get duration from voice clip if available
                    chunk_duration = voice_clips[i].duration if i < len(voice_clips) else 3  # Fallback 3 seconds

                    # Create simple white text clip
                    txt_clip = TextClip(
                        chunk,
                        fontsize=48,
                        color='white',
                        font=context.user_data.get('text_font', 'Times-Roman'),
                        size=(720, None),  # Width matches video width
                        method='caption'
                    ).set_duration(chunk_duration)

                    # Center the text
                    txt_clip = txt_clip.set_position(('center', 'center'))
                    txt_clip = txt_clip.set_start(current_start)

                    text_clips.append(txt_clip)
                    current_start += chunk_duration
                except Exception as e:
                    logger.error(f"Error creating text for chunk {i}: {e}")

        # Combine video and text
        if text_clips:
            final_video = CompositeVideoClip([final_video] + text_clips)

        # Mix audio tracks
        final_audio = []
        if voice_clips:
            voice_track = concatenate_audioclips(voice_clips)
            voice_track = voice_track.volumex(1.0)  # Full volume for voice
            final_audio.append(voice_track)

        # Add background music at reduced volume
        music_volume = 0.2 if voice_clips else 0.8
        music_clip = music_clip.volumex(music_volume)
        final_audio.append(music_clip)

        # Set final audio
        final_video = final_video.set_audio(CompositeAudioClip(final_audio))

        # Export final video
        output_path = os.path.join(video_storage, f"final_{update.message.from_user.id}.mp4")
        final_video.write_videofile(
            output_path,
            codec="libx264",
            audio_codec="aac",
            threads=4,
            preset="fast",
            ffmpeg_params=["-crf", "23", "-movflags", "+faststart"]
        )

        # Deduct credits and send result
        user_id = str(update.message.from_user.id)
        credits = load_credits()
        credits[user_id] = credits.get(user_id, 0) - 1
        save_credits(credits)

        await update.message.reply_video(
            video=open(output_path, 'rb'),
            caption=f"✨ Your video is ready! Credits left: {credits[user_id]}"
        )

    except Exception as e:
        logger.error(f"Final video error: {str(e)}")
        await update.message.reply_text("❌ Failed to create video. Please try again.")
    finally:
        # Cleanup resources
        try:
            if 'final_video' in locals():
                final_video.close()
            if 'music_clip' in locals():
                music_clip.close()
            for clip in voice_clips:
                clip.close()
            for clip in video_clips:
                clip.close()
            if 'output_path' in locals() and os.path.exists(output_path):
                os.remove(output_path)
        except Exception as e:
            logger.error(f"Cleanup error: {e}")

async def add_credits(update: Update, context: CallbackContext) -> None:
    """Admin command to add credits to a user."""
    try:
        _, password, user_id, amount = update.message.text.split()
        if hashlib.sha256(password.encode()).hexdigest() != ADMIN_PASSWORD_HASH:
            await update.message.reply_text("❌ Invalid admin password!")
            return

        credits = load_credits()
        old_credits = credits.get(user_id, 0)
        credits[user_id] = old_credits + int(amount)
        save_credits(credits)

        await update.message.reply_text(
            f"✅ Credits updated successfully!\n"
            f"User: {user_id}\n"
            f"Added: {amount} credits ⭐\n"
            f"Previous balance: {old_credits} credits\n"
            f"New balance: {credits[user_id]} credits ✨"
        )
    except ValueError:
        await update.message.reply_text(
            "❌ Usage: /add_credits <password> <user_id> <amount>\n"
            "Example: /add_credits your_password 123456789 5"
        )

async def start(update: Update, context: CallbackContext) -> None:
    """Send a welcome message when the command /start is issued."""
    user_id = str(update.message.from_user.id)
    credits = load_credits()
    current_credits = credits.get(user_id, 0)

    welcome_message = (
        "👋 Welcome to the Video Quote Creator Bot!\n\n"
        "Here's how to create your video:\n"
        "1️⃣ Send me one or more video clips\n"
        "2️⃣ Type your quote\n"
        "3️⃣ Choose text display style\n"
        "4️⃣ Add background music (MP3 or YouTube Shorts link)\n\n"
        "📝 Additional commands:\n"
        "/fonts - Choose a different font for your text\n\n"
        f"You have {current_credits} credits ⭐\n\n"
        "Let's start! Send me your first video clip 🎥"
    )

    # Reset user data
    context.user_data.clear()

    await update.message.reply_text(welcome_message)

def analyze_scene_for_reversal(clip: VideoFileClip) -> List[tuple[float, float]]:
    """Analyze video clip to find the best segments for reverse effect."""
    try:
        frames = []
        frame_times = np.arange(0, clip.duration, 1/clip.fps)
        last_frame = None

        # Collect frame metrics
        for t in frame_times:
            try:
                frame = clip.get_frame(t)
                # Convert frame to grayscale for simpler processing
                gray_frame = np.mean(frame, axis=2)

                # Calculate basic metrics
                intensity = float(np.mean(gray_frame))
                contrast = float(np.std(gray_frame))

                # Calculate motion
                if last_frame is not None:
                    # Calculate frame difference
                    diff = np.abs(gray_frame - last_frame)
                    motion = float(np.mean(diff))
                else:
                    motion = 0.0

                frames.append({
                    'intensity': intensity,
                    'contrast': contrast,
                    'motion': motion,
                    'time': t
                })

                last_frame = gray_frame

            except Exception as e:
                logger.warning(f"Skipping frame at time {t}: {str(e)}")
                if frames:
                    frames.append(frames[-1].copy())
                continue

        if not frames:
            return []

        # Find high-impact moments
        window_size = int(clip.fps * 0.5)  # Half second window
        potential_points = []

        for i in range(window_size, len(frames) - window_size):
            window = frames[i-window_size:i+window_size]

            # Calculate impact score using simple averages
            intensity_before = sum(f['intensity'] for f in window[:window_size]) / window_size
            intensity_after = sum(f['intensity'] for f in window[window_size:]) / window_size
            intensity_change = abs(intensity_before - intensity_after)

            motion_score = sum(f['motion'] for f in window) / len(window)

            contrast_before = sum(f['contrast'] for f in window[:window_size]) / window_size
            contrast_after = sum(f['contrast'] for f in window[window_size:]) / window_size
            contrast_change = abs(contrast_before - contrast_after)

            # Weighted score
            impact_score = (
                intensity_change * 0.3 +
                motion_score * 0.5 +     # Prioritize motion
                contrast_change * 0.2
            )

            if impact_score > SCENE_DETECTION_THRESHOLD:
                potential_points.append({
                    'time': frames[i]['time'],
                    'score': impact_score
                })

        # Sort and select best points
        potential_points.sort(key=lambda x: x['score'], reverse=True)
        best_points = potential_points[:2]  # Take top 2 points
        best_points.sort(key=lambda x: x['time'])

        # Create segments
        segments = []
        min_duration = 0.5

        for point in best_points:
            start_time = max(0, point['time'] - min_duration)
            end_time = min(clip.duration, point['time'] + min_duration)

            # Avoid overlapping segments
            if not segments or start_time >= segments[-1][1] + 0.2:
                segments.append((start_time, end_time))

        return segments

    except Exception as e:
        logger.error(f"Scene analysis failed: {str(e)}")
        return []

def create_reversed_segments(clip: VideoFileClip, segments: List[tuple[float, float]], music_path: str) -> VideoFileClip:
    """Create a new clip with reversed segments synchronized to music beats."""
    try:
        # Load music and detect beats
        y, sr = librosa.load(music_path)
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
        beat_times = librosa.frames_to_time(beat_frames, sr=sr)

        final_segments = []
        current_time = 0

        for start, end in segments:
            try:
                # Add original segment with error handling
                original = clip.subclip(start, max(0, min(end, clip.duration - 0.1)))
                if original.duration > 0.1:  # Only add if segment is valid
                    final_segments.append(original)

                    # Find nearest beat for transition
                    segment_end = current_time + original.duration
                    nearest_beat = beat_times[np.abs(beat_times - segment_end).argmin()]

                    # Create reversed segment
                    reversed_segment = time_mirror(original)
                    final_segments.append(reversed_segment)

                    current_time = segment_end + original.duration
            except Exception as e:
                logger.warning(f"Skipping segment {start}-{end}: {str(e)}")
                continue

        if not final_segments:
            return clip  # Return original clip if no valid segments

        return concatenate_videoclips(final_segments)
    except Exception as e:
        logger.error(f"Error in create_reversed_segments: {str(e)}")
        return clip  # Return original clip on error

def create_mixed_clips(video_clips: List[VideoFileClip], total_duration: float, music_path: str = None, use_reverse: bool = False) -> List[VideoFileClip]:
    """Create mixed clips with forward-reverse transitions on beats."""
    final_segments = []
    all_segments = []

    # Filter out any None or invalid clips
    video_clips = [clip for clip in video_clips if clip is not None and hasattr(clip, 'duration')]

    if not video_clips:
        logger.warning("No valid video clips to process")
        return []

    # Detect beats from music
    try:
        y, sr = librosa.load(music_path)
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
        beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
        avg_beat_duration = 60.0 / tempo if isinstance(tempo, (int, float)) else 60.0 / tempo[0]
        logger.info(f"Detected {len(beat_times)} beats, tempo: {tempo} BPM")
    except Exception as e:
        logger.error(f"Error analyzing music beats: {e}")
        avg_beat_duration = 0.5
        beat_times = []

    # Define segment durations based on beats
    min_segment_duration = 0.5  # Minimum 0.5 seconds
    max_segment_duration = 2.0  # Maximum 2 seconds

    # Process each clip to create random segments
    for clip_index, clip in enumerate(video_clips):
        try:
            # Verify clip is valid
            if not hasattr(clip, 'duration') or clip.duration < 0.1:
                logger.warning(f"Skipping invalid clip {clip_index}")
                continue

            # Get valid duration
            safe_duration = max(0, clip.duration - 0.1)
            if safe_duration < min_segment_duration:
                continue

            # Create multiple random segments from this clip
            num_segments = min(int(safe_duration / min_segment_duration), 10)  # Maximum 10 segments per clip

            for _ in range(num_segments):
                try:
                    # Choose random start time
                    max_start = safe_duration - min_segment_duration
                    start_time = random.uniform(0, max_start)

                    # Choose segment duration based on beats
                    if beat_times:
                        next_beats = [bt for bt in beat_times if bt > start_time]
                        if next_beats and next_beats[0] - start_time <= max_segment_duration:
                            segment_duration = next_beats[0] - start_time
                        else:
                            segment_duration = random.uniform(min_segment_duration, max_segment_duration)
                    else:
                        segment_duration = random.uniform(min_segment_duration, max_segment_duration)

                    # Ensure we don't exceed clip duration
                    end_time = min(start_time + segment_duration, safe_duration)
                    if end_time - start_time < min_segment_duration:
                        continue

                    # Create segment
                    segment = clip.subclip(start_time, end_time)

                    # Add normal segment
                    all_segments.append({
                        'forward': segment,
                        'reverse': None,
                        'duration': segment.duration,
                        'is_reverse_pair': False,
                        'start_time': start_time,
                        'clip_index': clip_index
                    })

                    # Create reverse segment if enabled
                    if use_reverse and segment.duration >= min_segment_duration:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            reverse_segment = time_mirror(segment)
                            all_segments.append({
                                'forward': segment.copy(),
                                'reverse': reverse_segment,
                                'duration': segment.duration * 2,
                                'is_reverse_pair': True,
                                'start_time': start_time,
                                'clip_index': clip_index
                            })

                except Exception as e:
                    logger.warning(f"Error creating segment from clip {clip_index}: {e}")
                    continue

        except Exception as e:
            logger.error(f"Error processing clip {clip_index}: {e}")
            continue

    if not all_segments:
        return video_clips

    # Thoroughly shuffle segments
    random.shuffle(all_segments)

    # Process segments with error handling
    try:
        current_time = 0
        last_clip_index = None
        last_was_reverse = False

        while current_time < total_duration and all_segments:
            segment_info = all_segments.pop(0)

            try:
                if segment_info['is_reverse_pair']:
                    if beat_times:
                        nearest_beat = min(beat_times, key=lambda x: abs(x - current_time))
                        if abs(nearest_beat - current_time) < 0.2:
                            current_time = nearest_beat

                    # Apply fade effects using function-style calls
                    forward_clip = fadein(fadeout(segment_info['forward'], 0.1), 0.1)
                    reverse_clip = fadein(fadeout(segment_info['reverse'], 0.1), 0.1)

                    final_segments.append(forward_clip)
                    final_segments.append(reverse_clip)
                    current_time += segment_info['duration']
                else:
                    # Apply fade effects to regular segment
                    clip_with_fade = fadein(fadeout(segment_info['forward'], 0.1), 0.1)
                    final_segments.append(clip_with_fade)
                    current_time += segment_info['duration']

                used_segments.append(segment_info)

            except Exception as e:
                logger.warning(f"Error adding segment: {e}")
                continue

        # Create final clip with error handling
        if not final_segments:
            return video_clips

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                final_clip = concatenate_videoclips(final_segments, method="compose")
                if final_clip.duration > total_duration:
                    final_clip = final_clip.subclip(0, total_duration - 0.1)
            return [final_clip]
        except Exception as e:
            logger.error(f"Error concatenating clips: {e}")
            return [video_clips[0]] if video_clips else []

    except Exception as e:
        logger.error(f"Error in segment processing: {e}")
        return [video_clips[0]] if video_clips else []

async def retry_on_timeout(func, *args, **kwargs):
    """Retry function on timeout with exponential backoff."""
    for attempt in range(MAX_RETRIES):
        try:
            return await func(*args, **kwargs)
        except (TimedOut, NetworkError) as e:
            if attempt == MAX_RETRIES - 1:  # Last attempt
                raise  # Re-raise the last exception
            wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
            logger.warning(f"Network error: {str(e)}. Retrying in {wait_time} seconds...")
            await asyncio.sleep(wait_time)

async def download_youtube_shorts(url: str) -> Optional[str]:
    """Download YouTube Shorts video without cookies dependency."""
    try:
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
            'outtmpl': os.path.join(video_storage, '%(id)s.%(ext)s'),
            'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'extractor_args': {
                'youtube': {
                    'player_client': ['android_embedded'],
                    'skip': ['dash', 'hls']
                }
            },
            'quiet': True
        }

        with YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            video_id = info['id']
            output_path = os.path.join(video_storage, f"{video_id}.mp3")

            if os.path.exists(output_path):
                return output_path

            ydl.download([url])
            return output_path if os.path.exists(output_path) else None

    except Exception as e:
        logger.error(f"YouTube download error: {str(e)}")
        return None

def improved_beat_detect(audio_path: str) -> List[float]:
    """
    Detect beats in the audio file using librosa.

    Args:
        audio_path (str): Path to the audio file.

    Returns:
        List[float]: List of beat times in seconds.
    """
    try:
        # Load audio file
        y, sr = librosa.load(audio_path, sr=None)

        # Detect tempo and beats
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)

        # Convert beat frames to time in seconds
        beat_times = librosa.frames_to_time(beat_frames, sr=sr)

        # Ensure at least one beat is detected
        if len(beat_times) == 0:
            logger.warning("No beats detected. Using fallback beat intervals.")
            # Fallback: Create evenly spaced beats based on tempo
            if tempo > 0:
                beat_interval = 60.0 / tempo  # Time between beats in seconds
                beat_times = np.arange(0, librosa.get_duration(y=y, sr=sr), beat_interval)
            else:
                # Default to 1 beat per second if tempo detection fails
                beat_times = np.arange(0, librosa.get_duration(y=y, sr=sr), 1.0)

        return beat_times.tolist()

    except Exception as e:
        logger.error(f"Error in beat detection: {e}")
        # Fallback: Return evenly spaced beats at 1-second intervals
        duration = librosa.get_duration(filename=audio_path)
        return list(np.arange(0, duration, 1.0))

async def main():
    """Main function to run the bot."""
    # Ensure setup is complete
    if not setup_fonts_and_imagemagick():
        logger.error("Failed to setup fonts and ImageMagick")
        return

    token = '7955939724:AAHKNQ-VqL8hgU5NOv783aHCsBciFVZ5-Cw'

    application = (
        Application.builder()
        .token(token)
        .connect_timeout(30.0)
        .read_timeout(30.0)
        .write_timeout(30.0)
        .pool_timeout(30.0)
        .get_updates_read_timeout(42.0)
        .get_updates_write_timeout(42.0)
        .get_updates_connect_timeout(42.0)
        .get_updates_pool_timeout(42.0)
        .build()
    )

    # Add handlers
    application.add_handler(CommandHandler("start", start))
    application.add_handler(CommandHandler("add_credits", add_credits))
    application.add_handler(MessageHandler(filters.VIDEO & ~filters.COMMAND, handle_video))
    application.add_handler(MessageHandler(
        filters.TEXT & filters.Regex(r'(https?://)?(www\.)?(youtube\.com/shorts/|youtu\.be/)') & ~filters.COMMAND,
        handle_music
    ))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_text))
    application.add_handler(MessageHandler(filters.AUDIO & ~filters.COMMAND, handle_music))

    # Replace font handlers with new ones
    application.add_handler(CommandHandler("fonts", show_fonts))
    application.add_handler(CallbackQueryHandler(font_callback, pattern="^font_"))
    application.add_handler(MessageHandler(filters.Document.ALL & ~filters.COMMAND, handle_font_file))

    print("Bot started at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("Bot will run until:", (datetime.now() + timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S"))

    try:
        # Run the application
        await application.run_polling(
            allowed_updates=Update.ALL_TYPES,
            drop_pending_updates=True,
            poll_interval=1.0,
            timeout=30,
            close_loop=False  # Don't try to close the loop
        )
    except Exception as e:
        logger.error(f"Error running bot: {str(e)}")
        raise
    finally:
        print("Bot stopped at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        # Cleanup temporary files
        try:
            temp_files = [f for f in os.listdir(video_storage) if f.endswith(('.mp4', '.mp3'))]
            for f in temp_files:
                try:
                    os.remove(os.path.join(video_storage, f))
                except:
                    pass
        except:
            pass

if __name__ == '__main__':
    # Apply nest_asyncio to fix event loop issues
    nest_asyncio.apply()

    # Get the current event loop
    loop = asyncio.get_event_loop()

    try:
        # Run the main function
        loop.run_until_complete(main())
    except KeyboardInterrupt:
        print("\nBot stopped by user")
    except Exception as e:
        print(f"Error: {e}")